# ch07 Bonus 02：指令模型评估

> 对照官方 `ch07/03_model-evaluation`

## 一句话

用一组测试指令评估模型，靠**本地启发式指标**（响应长度、重复度、是否相关）打分，不依赖外部 API。

## 背景

官方用 GPT-4 / Llama3 API 当「裁判」给模型回答打分。但本地无 API 时，可用启发式指标做粗略评估：

- **响应长度**：过短可能敷衍，过长可能跑题
- **重复度**：n-gram 重复率高说明模型在「复读」
- **关键词命中**：回答是否包含期望的关键词（如「巴黎」）

> 这套指标不能完全替代人工/LM-as-judge，但能快速发现明显问题。

In [ ]:
import torch
import torch.nn.functional as F
import tiktoken
from src.gpt import GPTModel, GPT_CONFIG_124M

# 构造测试指令集（含期望关键词，用于自动评估）
TEST_CASES = [
    {"instruction": "识别情感", "input": "今天天气真好", "expected_keyword": "正"},
    {"instruction": "翻译成英文", "input": "你好", "expected_keyword": "Hello"},
    {"instruction": "回答问题", "input": "法国首都是哪", "expected_keyword": "巴黎"},
]

# 复用主线的训练逻辑（快速训练一个 demo 模型）
def format_prompt(entry):
    return (f"### Instruction:\n{entry['instruction']}\n"
            f"### Input:\n{entry['input']}\n### Response:\n")

tok = tiktoken.get_encoding("gpt2")
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 64})
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 用主线同款数据训练
TRAIN_DATA = [
    {"instruction": "识别情感", "input": "今天天气真好", "output": " 正面"},
    {"instruction": "识别情感", "input": "太让人失望了", "output": " 负面"},
    {"instruction": "翻译成英文", "input": "你好", "output": " Hello"},
    {"instruction": "翻译成英文", "input": "谢谢", "output": " Thank you"},
    {"instruction": "回答问题", "input": "法国首都是哪", "output": " 巴黎"},
] * 5

torch.manual_seed(123)
model = GPTModel(cfg)
for p in model.parameters(): p.requires_grad = False
for p in model.trf_blocks[-1].parameters(): p.requires_grad = True
for p in model.final_norm.parameters(): p.requires_grad = True
for p in model.out_head.parameters(): p.requires_grad = True
model.to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4)

def collate(batch):
    xs, ys = [], []
    for e in batch:
        p = tok.encode(format_prompt(e)); r = tok.encode(e["output"])
        ids = (p+r)[:64]; t = (ids[1:]+[tok.eot_token])[:64]
        for i in range(min(len(p), len(t))): t[i] = -100
        ids = ids + [50256]*(64-len(ids)); t = t + [-100]*(64-len(t))
        xs.append(ids); ys.append(t)
    return torch.tensor(xs), torch.tensor(ys)

model.train()
for _ in range(10):
    for i in range(0, len(TRAIN_DATA), 4):
        x, y = collate(TRAIN_DATA[i:i+4])
        opt.zero_grad()
        F.cross_entropy(model(x.to(device)).flatten(0,1), y.to(device).flatten(),
                       ignore_index=-100).backward()
        opt.step()
print("✓ demo 模型已训练")

In [ ]:
# 生成函数
def generate(entry, max_new=8):
    model.eval()
    idx = torch.tensor([tok.encode(format_prompt(entry))]).to(device)
    with torch.no_grad():
        for _ in range(max_new):
            logits = model(idx[:, -cfg["context_length"]:])[:, -1, :]
            nid = logits.argmax(-1, keepdim=True)
            idx = torch.cat([idx, nid], dim=1)
            if nid.item() == tok.eot_token: break
    return tok.decode(idx[0].tolist()).split("### Response:\n")[-1].strip()

# 启发式评估指标
def evaluate_response(response, expected_keyword):
    """返回 (关键词命中, 长度, 重复度)。"""
    keyword_hit = expected_keyword in response
    length = len(response)
    # 简单重复度：字符级 unique 比例
    chars = list(response)
    uniqueness = len(set(chars)) / max(len(chars), 1)  # 越高越多样
    return keyword_hit, length, uniqueness

# 评估
print(f"{'指令':<10} {'输入':<14} {'生成':<16} {'关键词':<8} {'长度':<5} {'多样性':<7}")
print("-" * 68)
for tc in TEST_CASES:
    resp = generate(tc)
    hit, length, uniq = evaluate_response(resp, tc["expected_keyword"])
    hit_str = "✓命中" if hit else "✗未中"
    print(f"{tc['instruction']:<10} {tc['input']:<14} {resp:<16} {hit_str:<8} {length:<5} {uniq:.2f}")

print("\n💡 demo 模型未预训练，关键词命中率低。加载预训练权重 + 更多数据后会显著提升。")
print("   多样性 < 0.5 说明模型在复读；长度过短可能敷衍。")